In [1]:
using CMPSExcitations

In [2]:
# canonical basis
function projection_matrix(D, R)
    Dr, M = eigen(R)

    dim_in = D^2 + D      # input: E (DxD) + F (D)
    dim_out = 2 * D^2     # output: W1, W2 (both DxD)
    P = zeros(ComplexF64, dim_out, dim_in)

    for j in 1:dim_in
        e = zeros(Float64, dim_in)
        e[j] = 1.0

        E = reshape(view(e, 1:D^2), D, D)
        F = view(e, D^2+1:dim_in)

        W1 = E
        W2 = E + M * Diagonal(F) / M

        P[:, j] = vcat(vec(W1), vec(W2))
    end

    return P
end

# canonical basis
function excitation_matrix(Heff, D)
    dim = 2 * D^2
    M = zeros(ComplexF64, dim, dim)

    for j in 1:dim
        e = zeros(ComplexF64, dim)
        e[j] = 1.0
        W1 = reshape(view(e, 1:D^2), D, D)
        W2 = reshape(view(e, D^2+1:dim), D, D)
        W1p, W2p = Heff((Constant(W1), Constant(W2)))
        M[:, j] = vcat(vec(W1p[]), vec(W2p[]))
    end

    return M
end

function excitation_matrix_constrained(Heff, M)
    D = size(M, 1) # R = MDᵣ/M
    H = excitation_matrix(Heff, D)
    P = projection_matrix(D, M)

    P' * H * P, P
end

excitation_matrix_constrained (generic function with 1 method)

In [3]:
Hsingle_ll(c, μ) = ∫(∂ψ̂' * ∂ψ̂ - μ * ψ̂' * ψ̂ + c * (ψ̂')^2 * ψ̂^2, (-Inf, +Inf));
Hsingle(c, μ) = ∫(2 * ∂ψ̂' * ∂ψ̂ - 2 * μ * ψ̂' * ψ̂ + 4 * c * (ψ̂')^2 * ψ̂^2, (-Inf, +Inf));
Hcoupled(c, μ) = ∫(
    (∂ψ̂₁' * ∂ψ̂₁ - μ * ψ̂₁' * ψ̂₁ + c * (ψ̂₁')^2 * ψ̂₁^2 +
     ∂ψ̂₂' * ∂ψ̂₂ - μ * ψ̂₂' * ψ̂₂ + c * (ψ̂₂')^2 * ψ̂₂^2 +
     2 * c * (ψ̂₁') * (ψ̂₂') * ψ̂₂ * ψ̂₁), (-Inf, +Inf));

In [4]:
c, μ = 10., 5.
tol = 1e-10

Ds = [4, 8]
D = maximum(Ds)

Niter = 10
states = Vector{InfiniteCMPS}(undef, Niter)
energies = zeros(ComplexF64, Niter)

Threads.@threads for n in 1:Niter
    # HLL = Hsingle(c, μ)
    # @time stateLL = find_groundstate(Ds, HLL, YangGaudinCMPS, optalg=LBFGS(80; verbosity=1, maxiter=7000, gradtol=tol), gradtol=tol)
    # println("Energy density: ", expval(HLL.h, stateLL)[], "\n Particle density: ", expval(ψ̂' * ψ̂, stateLL)[], "\n Order parameter: ", expval(ψ̂, stateLL)[])
    # stateCLL = InfiniteCMPS(stateLL.Q, (stateLL.Rs[1], stateLL.Rs[1]))

    HLL = Hsingle_ll(c, μ)
    @time stateLL = find_groundstate(Ds, HLL, InfiniteCMPS, optalg=LBFGS(80; verbosity=1, maxiter=7000, gradtol=tol), gradtol=tol)
    println("Energy density: ", expval(HLL.h, stateLL)[], "\n Particle density: ", expval(ψ̂' * ψ̂, stateLL)[], "\n Order parameter: ", expval(ψ̂, stateLL)[])
    stateCLL = InfiniteCMPS(stateLL.Q, (stateLL.Rs[1] / sqrt(2), stateLL.Rs[1] / sqrt(2)))

    HCLL = Hcoupled(c, μ)
    states[n] = stateCLL
    energies[n] = expval(HCLL.h, stateCLL)[]
end

Optimizing D=4
Optimizing D=4
Optimizing D=4
Optimizing D=4
Optimizing D=4
Optimizing D=4
Optimizing D=4
Optimizing D=4
Optimizing D=4
Optimizing D=4


┌ Warning: The function `inner` is not implemented for (values of) type `Tuple{Constant{Matrix{Float64}}, Constant{Matrix{Float64}}}`;
│ this fallback will disappear in future versions of VectorInterface.jl
└ @ VectorInterface /home/ashankar/.julia/packages/VectorInterface/J6qCR/src/fallbacks.jl:196
┌ Warning: The function `add!!` is not implemented for (values of) type `Tuple{Constant{Matrix{Float64}}, Constant{Matrix{Float64}}, Float64, VectorInterface.One}`;
│ this fallback will disappear in future versions of VectorInterface.jl
└ @ VectorInterface /home/ashankar/.julia/packages/VectorInterface/J6qCR/src/fallbacks.jl:163
┌ Warning: The function `scale` is not implemented for (values of) type `Tuple{Constant{Matrix{Float64}}, Float64}`;
│ this fallback will disappear in future versions of VectorInterface.jl
└ @ VectorInterface /home/ashankar/.julia/packages/VectorInterface/J6qCR/src/fallbacks.jl:67
┌ Warning: The function `scale!!` is not implemented for (values of) type `Tuple{Const

D = 4 | InfiniteCMPS{Constant{Matrix{Float64}}, 1}
D = 4 | InfiniteCMPS{Constant{Matrix{Float64}}, 1}
D = 4 | InfiniteCMPS{Constant{Matrix{Float64}}, 1}
D = 4 | InfiniteCMPS{Constant{Matrix{Float64}}, 1}
D = 4 | InfiniteCMPS{Constant{Matrix{Float64}}, 1}
D = 4 | InfiniteCMPS{Constant{Matrix{Float64}}, 1}
D = 4 | InfiniteCMPS{Constant{Matrix{Float64}}, 1}
D = 4 | InfiniteCMPS{Constant{Matrix{Float64}}, 1}
D = 4 | InfiniteCMPS{Constant{Matrix{Float64}}, 1}
 13.687316 seconds (61.68 M allocations: 3.035 GiB, 3.01% gc time, 1039043 lock conflicts, 906.09% compilation time: <1% of which was recompilation)
 13.687008 seconds (61.68 M allocations: 3.034 GiB, 3.01% gc time, 1039042 lock conflicts, 906.11% compilation time: <1% of which was recompilation)
  8.922840 seconds (36.03 M allocations: 1.754 GiB, 3.00% gc time, 1039036 lock conflicts, 934.95% compilation time: <1% of which was recompilation)
 13.687055 seconds (61.68 M allocations: 3.035 GiB, 3.01% gc time, 1039041 lock conflicts, 906

┌ Info: UniformCMPS ground state: converged after 108 iterations: e = -2.734747817523, ‖∇e‖ = 1.9136e-11
└ @ CMPSKit /home/ashankar/Documents/PhDstuff/code/TensorNetworks/CMPSKit.jl/src/infinitecmps/groundstate.jl:127


D = 4 | InfiniteCMPS{Constant{Matrix{Float64}}, 1}
 12.807289 seconds (55.17 M allocations: 2.709 GiB, 3.00% gc time, 1039057 lock conflicts, 950.43% compilation time: <1% of which was recompilation)
---------------
Optimizing D=8


...

D = 8 | InfiniteCMPS{Constant{Matrix{Float64}}, 1}
 73.240039 seconds (39.88 M allocations: 3.323 GiB, 0.45% gc time, 66862620 lock conflicts, 14.89% compilation time)
---------------
 86.949287 seconds (101.65 M allocations: 6.362 GiB, 0.86% gc time, 67901675 lock conflicts, 155.39% compilation time: <1% of which was recompilation)
Energy density: -2.7612655090868325
 Particle density: 0.8729544307598076
 Order parameter: 0.511577282919531


┌ Info: LBFGS: converged after 324 iterations: f = -2.761265509087, ‖∇f‖ = 9.9428e-11
└ @ OptimKit /home/ashankar/.julia/packages/OptimKit/xpmbV/src/lbfgs.jl:138
┌ Info: UniformCMPS ground state: converged after 325 iterations: e = -2.761265509087, ‖∇e‖ = 9.9428e-11
└ @ CMPSKit /home/ashankar/Documents/PhDstuff/code/TensorNetworks/CMPSKit.jl/src/infinitecmps/groundstate.jl:127


D = 8 | InfiniteCMPS{Constant{Matrix{Float64}}, 1}
 73.480385 seconds (40.04 M allocations: 3.338 GiB, 0.44% gc time, 66887983 lock conflicts, 14.84% compilation time)
---------------
 87.729953 seconds (106.35 M allocations: 6.608 GiB, 0.89% gc time, 67927045 lock conflicts, 157.10% compilation time: <1% of which was recompilation)


┌ Info: LBFGS: converged after 342 iterations: f = -2.761265509087, ‖∇f‖ = 6.1027e-11
└ @ OptimKit /home/ashankar/.julia/packages/OptimKit/xpmbV/src/lbfgs.jl:138
┌ Info: UniformCMPS ground state: converged after 343 iterations: e = -2.761265509087, ‖∇e‖ = 6.1027e-11
└ @ CMPSKit /home/ashankar/Documents/PhDstuff/code/TensorNetworks/CMPSKit.jl/src/infinitecmps/groundstate.jl:127


Energy density: -2.7612655090868654
 Particle density: 0.8729544307607413
 Order parameter: 0.511577282912648


In [5]:
function leftcanonical(state, cmps=false)
    leftgauge!(state)
    r = rightenv(state)[1][]
    D, U = eigen(r)
    Q = U \ state.Q[] * U
    R = U \ state.Rs[1][] * U
    return (cmps) ? InfiniteCMPS(Constant(Q), (Constant(R), Constant(R))) : (Q, R)
end

leftcanonical (generic function with 2 methods)

In [6]:
trivial_spectrum = zeros(ComplexF64, Niter);
HCLL = Hcoupled(c, μ)

for idx in 1:Niter
    stateCLL = states[idx]
    stateCLL = leftcanonical(states[idx], true)
    space = InfiniteCMPSExcitationSpace(0, stateCLL, stateCLL)
    Heff, P = excitation_matrix_constrained(excitation_operator(HCLL, space), eigen(stateCLL.Rs[1][]).vectors)

    vals, vecs = eigsolve(Heff, rand(size(Heff, 2)), 5, :SR, maxiter=200)
    trivial_spectrum[idx] = vals[1]
end

┌ Warning: The function `add` is not implemented for (values of) type `Tuple{Constant{Matrix{ComplexF64}}, Constant{Matrix{ComplexF64}}, ComplexF64, ComplexF64}`;
│ this fallback will disappear in future versions of VectorInterface.jl
└ @ VectorInterface /home/ashankar/.julia/packages/VectorInterface/J6qCR/src/fallbacks.jl:131


In [7]:
energies

10-element Vector{ComplexF64}:
  -2.761265509086993 + 0.0im
 -2.7612655090871177 + 0.0im
  -2.761265509087037 + 0.0im
 -2.7612655090870692 + 0.0im
 -2.7612655090870972 + 0.0im
 -2.7612655090869205 + 0.0im
  -2.761265509087069 + 0.0im
 -2.7612655090870684 + 0.0im
 -2.7612655090870666 + 0.0im
  -2.761265509087154 + 0.0im

In [9]:
trivial_spectrum

10-element Vector{ComplexF64}:
    0.07668624163246242 + 1.0952911993792193e-11im
  -0.051320819261944095 + 3.836603519833025e-12im
    0.39992744932622143 + 1.7962924573175907e-12im
   -0.16162430547695095 - 1.0254945598247594e-11im
   -0.17352821964539566 - 1.610363728212176e-12im
 -0.0004936019152855194 + 2.6521099078278594e-12im
    0.18873195539322693 + 2.63271514873997e-12im
    0.20212915273786075 - 8.648939487875296e-13im
   -0.14934541577016522 - 2.4950755470559452e-12im
    0.01912078782902136 - 1.1721841625041051e-12im

In [106]:
leftcanonical(states[2])[2]

8×8 Matrix{Float64}:
  0.261541  -0.169013    0.0615951   0.0403976  …  -0.00442266   -0.000825746
  0.299566  -0.299128    0.115868   -0.0413388     -0.000857989   0.00873781
 -0.219437   0.232891    0.215729   -0.22425        0.0405453     0.000724789
  0.366592   0.211647    0.571211   -0.279009      -0.0208285    -0.0408231
  0.023395   0.223977    0.363198   -0.313292       0.147972     -0.0097222
 -1.15673    0.0707694  -0.532027    0.42528    …  -0.107343      0.131086
 -0.407785   0.0446331  -1.04936    -0.21163       -0.244818     -0.335181
  0.189492   1.13129     0.0466866   1.03234        0.834212      0.477658

In [107]:
leftcanonical(states[1])[2]

8×8 Matrix{Float64}:
 -0.261541  -0.169013    0.0615951   0.0403976  …   0.00442266    0.000825746
  0.299566   0.299128   -0.115868    0.0413388     -0.000857989   0.00873781
 -0.219437  -0.232891   -0.215729    0.22425        0.0405453     0.000724789
  0.366592  -0.211647   -0.571211    0.279009      -0.0208285    -0.0408231
 -0.023395   0.223977    0.363198   -0.313292      -0.147972      0.0097222
  1.15673    0.0707694  -0.532027    0.42528    …   0.107343     -0.131086
  0.407785   0.0446331  -1.04936    -0.21163        0.244818      0.335181
 -0.189492   1.13129     0.0466866   1.03234       -0.834212     -0.477658

In [108]:
(leftcanonical(states[2])[1] ./ leftcanonical(states[1])[1])

8×8 Matrix{Float64}:
  1.0  -1.0  -1.0  -1.0   1.0   1.0   1.0   1.0
 -1.0   1.0   1.0   1.0  -1.0  -1.0  -1.0  -1.0
 -1.0   1.0   1.0   1.0  -1.0  -1.0  -1.0  -1.0
 -1.0   1.0   1.0   1.0  -1.0  -1.0  -1.0  -1.0
  1.0  -1.0  -1.0  -1.0   1.0   1.0   1.0   1.0
  1.0  -1.0  -1.0  -1.0   1.0   1.0   1.0   1.0
  1.0  -1.0  -1.0  -1.0   1.0   1.0   1.0   1.0
  1.0  -1.0  -1.0  -1.0   1.0   1.0   1.0   1.0

In [109]:
(leftcanonical(states[2])[2] ./ leftcanonical(states[1])[2])

8×8 Matrix{Float64}:
 -1.0   1.0   1.0   1.0  -1.0  -1.0  -1.0  -1.0
  1.0  -1.0  -1.0  -1.0   1.0   1.0   1.0   1.0
  1.0  -1.0  -1.0  -1.0   1.0   1.0   1.0   1.0
  1.0  -1.0  -1.0  -1.0   1.0   1.0   1.0   1.0
 -1.0   1.0   1.0   1.0  -1.0  -1.0  -1.0  -1.0
 -1.0   1.0   1.0   1.0  -1.0  -1.0  -1.0  -1.0
 -1.0   1.0   1.0   1.0  -1.0  -1.0  -1.0  -1.0
 -1.0   1.0   1.0   1.0  -1.0  -1.0  -1.0  -1.0

In [115]:
rightenv(states[1])[1][]

8×8 Matrix{Float64}:
  0.491563     -0.347722     0.268566   …  -0.0120161   -0.0140619
 -0.347722      0.307011    -0.271593       0.0125564    0.0124515
  0.268566     -0.271593     0.330833      -0.0224981   -0.024514
  0.00946933   -0.0341041    0.0774354     -0.00665022  -0.00804807
  0.0123046    -0.0104116    0.014135      -0.00147955  -0.00153907
  0.000789581   0.00443009  -0.0169712  …   0.00232093   0.00262224
 -0.0120161     0.0125564   -0.0224981      0.00290493   0.00281056
 -0.0140619     0.0124515   -0.024514       0.00281056   0.00335649

In [116]:
rightenv(states[2])[1][]

8×8 Matrix{Float64}:
  0.332011     -0.0600466    -0.108935    …  -0.00959103    0.000835671
 -0.0600466     0.0186721     0.0279096       0.00410511   -0.000316011
 -0.108935      0.0279096     0.139122        0.0141472     0.00242526
 -0.423232      0.0872084     0.0581861       0.00899499   -0.00537546
 -0.0116885     0.00393528    0.00511459      0.00090946    2.78363e-5
 -0.0237147     0.00792237    0.0233459   …   0.00355727    0.000472965
 -0.00959103    0.00410511    0.0141472       0.00223309    0.000213676
  0.000835671  -0.000316011   0.00242526      0.000213676   0.000286803

In [114]:
eigen(rightenv(states[1])[1][]).vectors

8×8 Matrix{Float64}:
  0.00700162   0.0614801  -0.00863388  …  -0.444036   -0.589866    0.661546
 -0.0111478    0.0946565   0.0156969      -0.779314   -0.0153578  -0.545855
 -0.00900995   0.051793    0.072462       -0.200393    0.687375    0.507516
  0.00497162   0.0246044  -0.204171       -0.228873    0.395784    0.0678539
 -0.926883    -0.124608   -0.225662       -0.0833801   0.0270306   0.0216223
  0.0532462   -0.485945   -0.625144    …   0.132614   -0.100018   -0.0113759
 -0.325653    -0.184764    0.670217        0.148239   -0.0747795  -0.0272479
 -0.178111     0.835563   -0.248643        0.237587   -0.0797929  -0.0297124

In [113]:
eigen(rightenv(states[2])[1][]).vectors

8×8 Matrix{Float64}:
  0.00319044   0.0760046  -0.0673812  …   0.712093   0.296318    0.557861
  0.0175616    0.0661495   0.187392       0.398673  -0.0819649  -0.111855
  0.00504924   0.0486631  -0.0929999      0.307692  -0.866071   -0.132403
 -0.00907077   0.0544482  -0.0467092      0.366778   0.364496   -0.81024
 -0.200823    -0.443174   -0.820534       0.104241  -0.0123841  -0.0226517
  0.488377     0.110471    0.0497698  …   0.244633  -0.123921   -0.038606
 -0.665348    -0.403823    0.470366       0.182302  -0.0802617  -0.0153619
 -0.527299     0.782838   -0.229165      -0.028931  -0.0242837   0.00457446

In [117]:
energies

10-element Vector{ComplexF64}:
 -2.7612655090870364 + 0.0im
  -2.761265509087011 + 0.0im
  -2.761265509087132 + 0.0im
 -2.7612655090870537 + 0.0im
  -2.761265509087015 + 0.0im
  -2.761265509087113 + 0.0im
 -2.7612655090870386 + 0.0im
  -2.761265509087074 + 0.0im
  -2.761265509086975 + 0.0im
 -2.7612655090870004 + 0.0im

In [118]:
# matrices are equivalent upto gauge (and some sign..)
sum([norm(abs.(leftcanonical(states[i])[1]) .- abs.(leftcanonical(states[j])[1])) for i in 1:Niter for j in 1:i])

9.004517940735592e-8

In [144]:
let
    Ps = []
    for i in eachindex(states)
        R = states[i].Rs[1][]
        M = eigen(R).vectors
        push!(Ps, projection_matrix(size(R, 1), M))
    end

    display(Ps[1])
    display(Ps[2])
end

128×72 Matrix{ComplexF64}:
 1.0+0.0im  0.0+0.0im  0.0+0.0im  0.0+0.0im  …          0.0+0.0im
 0.0+0.0im  1.0+0.0im  0.0+0.0im  0.0+0.0im             0.0+0.0im
 0.0+0.0im  0.0+0.0im  1.0+0.0im  0.0+0.0im             0.0+0.0im
 0.0+0.0im  0.0+0.0im  0.0+0.0im  1.0+0.0im             0.0+0.0im
 0.0+0.0im  0.0+0.0im  0.0+0.0im  0.0+0.0im             0.0+0.0im
 0.0+0.0im  0.0+0.0im  0.0+0.0im  0.0+0.0im  …          0.0+0.0im
 0.0+0.0im  0.0+0.0im  0.0+0.0im  0.0+0.0im             0.0+0.0im
 0.0+0.0im  0.0+0.0im  0.0+0.0im  0.0+0.0im             0.0+0.0im
 0.0+0.0im  0.0+0.0im  0.0+0.0im  0.0+0.0im             0.0+0.0im
 0.0+0.0im  0.0+0.0im  0.0+0.0im  0.0+0.0im             0.0+0.0im
    ⋮                                        ⋱  
 0.0-0.0im  0.0-0.0im  0.0-0.0im  0.0-0.0im       0.0085357+0.00974785im
 0.0-0.0im  0.0-0.0im  0.0-0.0im  0.0-0.0im  …    0.0719098-0.168637im
 0.0-0.0im  0.0-0.0im  0.0-0.0im  0.0-0.0im       -0.049988+0.105583im
 0.0-0.0im  0.0-0.0im  0.0-0.0im  0.0-0.0im      

128×72 Matrix{ComplexF64}:
 1.0+0.0im  0.0+0.0im  0.0+0.0im  0.0+0.0im  …         0.0+0.0im
 0.0+0.0im  1.0+0.0im  0.0+0.0im  0.0+0.0im            0.0+0.0im
 0.0+0.0im  0.0+0.0im  1.0+0.0im  0.0+0.0im            0.0+0.0im
 0.0+0.0im  0.0+0.0im  0.0+0.0im  1.0+0.0im            0.0+0.0im
 0.0+0.0im  0.0+0.0im  0.0+0.0im  0.0+0.0im            0.0+0.0im
 0.0+0.0im  0.0+0.0im  0.0+0.0im  0.0+0.0im  …         0.0+0.0im
 0.0+0.0im  0.0+0.0im  0.0+0.0im  0.0+0.0im            0.0+0.0im
 0.0+0.0im  0.0+0.0im  0.0+0.0im  0.0+0.0im            0.0+0.0im
 0.0+0.0im  0.0+0.0im  0.0+0.0im  0.0+0.0im            0.0+0.0im
 0.0+0.0im  0.0+0.0im  0.0+0.0im  0.0+0.0im            0.0+0.0im
    ⋮                                        ⋱  
 0.0-0.0im  0.0-0.0im  0.0-0.0im  0.0-0.0im     0.00399639+0.0485954im
 0.0-0.0im  0.0-0.0im  0.0-0.0im  0.0-0.0im  …   -0.415616-1.40324im
 0.0-0.0im  0.0-0.0im  0.0-0.0im  0.0-0.0im      0.0653987+0.195979im
 0.0-0.0im  0.0-0.0im  0.0-0.0im  0.0-0.0im      -0.340274-1.885